# 05 — Full Backtest + SHAP Report
Run walk-forward backtest across all targets × horizons × models. Generate final metrics table and SHAP plots for README/dashboard.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.features import build_feature_matrix
from src.backtest import walk_forward_backtest
from src.evaluate import compile_metrics, save_metrics, plot_predictions, plot_shap_summary
from src.models.baseline import ARIMAWrapper, NaiveLastValue
from src.models.ml_models import LightGBMForecaster

df_raw = pd.read_csv('../data/raw/macro_raw.csv', index_col='date', parse_dates=True)
print('Setup complete')

In [ ]:
TARGETS = ['cpi', 'unemployment']  # skip gdp for speed (quarterly = fewer samples)
HORIZONS = [1, 2, 4]
MODELS = [
    ('Naive', NaiveLastValue),
    ('ARIMA', lambda: ARIMAWrapper(order=(2, 0, 1))),
    ('LightGBM', LightGBMForecaster),
]

all_results = []

for target in TARGETS:
    for horizon in HORIZONS:
        X, y = build_feature_matrix(df_raw, target_col=target, forecast_horizon=horizon)
        print(f'\n── {target.upper()} h={horizon} ({X.shape[1]} features) ──')
        for name, ModelClass in MODELS:
            model = ModelClass()
            result = walk_forward_backtest(X, y, model, model_name=name, horizon=horizon)
            all_results.append(result)
            print(f'  {name:12s}  RMSE={result.rmse:.4f}  MAE={result.mae:.4f}')

print('\n✓ All backtests done')

In [ ]:
# Final metrics table
final_metrics = compile_metrics(all_results)
save_metrics(final_metrics)
final_metrics.pivot_table(index=['horizon', 'model'], values='rmse').round(4)

In [ ]:
# Bar chart comparison for README
for target in TARGETS:
    target_results = [r for r in all_results if r.horizon == 1]
    names = [r.model_name for r in target_results]
    rmses = [r.rmse for r in target_results]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(names, rmses, color=['#aec6cf', '#ffb347', '#77dd77'])
    ax.set_title(f'RMSE by model — {target.upper()} (h=1)', fontsize=12)
    ax.set_ylabel('RMSE')
    for bar, val in zip(bars, rmses):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'../outputs/figures/model_comparison_{target}.png', dpi=150)
    plt.show()

In [ ]:
# SHAP for LightGBM CPI h=1 (most interpretable for README)
X_shap, y_shap = build_feature_matrix(df_raw, target_col='cpi', forecast_horizon=1)
lgbm = LightGBMForecaster()
lgbm.fit(X_shap, y_shap)
plot_shap_summary(lgbm._model, X_shap, model_name='final_lgbm_cpi_h1')
print('All figures saved to outputs/figures/')